# Import Libraries

In [1]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import audiomentations as A
from audiomentations import AddGaussianNoise, TimeStretch, PitchShift
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Loading Data

## TESS Data

In [2]:
tess_path = '../Data/TESS Toronto emotional speech set data/TESS Toronto emotional speech set data'

In [3]:
tess_file_paths = []
tess_labels = []

for folder in os.listdir(tess_path):
    folder_path = os.path.join(tess_path, folder)
    label = folder[4:]
    label = label.lower() 

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        tess_file_paths.append(file_path)
        tess_labels.append(label)

In [4]:
tess_data = pd.DataFrame({
    'paths' : tess_file_paths,
    'emotions' : tess_labels
})

tess_data.head()

,paths,emotions
0,../Data/TESS Toronto emotional speech set data...,angry
1,../Data/TESS Toronto emotional speech set data...,angry
2,../Data/TESS Toronto emotional speech set data...,angry
3,../Data/TESS Toronto emotional speech set data...,angry
4,../Data/TESS Toronto emotional speech set data...,angry


## SAVEE Data

In [5]:
base_path = "../Data/SAVEE data/SAVEE data"
speakers = ["DC", "JE", "JK", "KL"]
savee_paths = [os.path.join(base_path, speaker) for speaker in speakers]

In [6]:
emotion_mapping = {
    'a': 'angry',
    'd': 'disgust',
    'h': 'happy',
    'sa': 'sad',
    'su': 'pleasant_surprise',
    'f': 'fearful',
    'n': 'neutral'
}

In [7]:
savee_file_paths = []
savee_labels = []

for path in savee_paths:
    if not os.path.exists(path):
        print(f"Skipping missing directory: {path}")
        continue
    
    for file in os.listdir(path):
        if file.endswith(".wav"):
            emotion_key = ''.join(filter(str.isalpha, file.split('.')[0]))  # Extract emotion prefix
            if emotion_key in emotion_mapping:
                file_path = os.path.join(path, file)
                savee_file_paths.append(file_path)
                savee_labels.append(emotion_mapping[emotion_key])

In [8]:
savee_data = pd.DataFrame({
    'paths': savee_file_paths,
    'emotions': savee_labels
})

savee_data.head()

,paths,emotions
0,../Data/SAVEE data/SAVEE data\DC\a01.wav,angry
1,../Data/SAVEE data/SAVEE data\DC\a02.wav,angry
2,../Data/SAVEE data/SAVEE data\DC\a03.wav,angry
3,../Data/SAVEE data/SAVEE data\DC\a04.wav,angry
4,../Data/SAVEE data/SAVEE data\DC\a05.wav,angry


# Data augmentation

## TESS Data

In [9]:
random.seed(42)
np.random.seed(42)

augment = A.Compose([
    A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    A.PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    A.TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5)
])  

augmented_dir = "../Data/augmented_tess"
os.makedirs(augmented_dir, exist_ok=True)


X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    tess_file_paths, tess_labels, test_size=0.1, random_state=42, stratify=tess_labels
)

augmented_file_paths = []
augmented_labels = []

for file_path, label in zip(X_train_paths, y_train):
    try:
        audio, sr = librosa.load(file_path, sr=None)
        
        augmented_file_paths.append(file_path)
        augmented_labels.append(label)

        augmented_audio = augment(audio, sample_rate=sr)

        augmented_file_path = os.path.join(augmented_dir, os.path.basename(file_path).replace('.wav', '_augmented.wav'))
        sf.write(augmented_file_path, augmented_audio, sr)
        augmented_file_paths.append(augmented_file_path)
        augmented_labels.append(label)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

train_data = pd.DataFrame({
    'paths': augmented_file_paths,
    'emotions': augmented_labels
})
test_data = pd.DataFrame({
    'paths': X_test_paths,
    'emotions': y_test
})

print("Unique train paths:", len(set(train_data['paths'])))
print("Unique test paths:", len(set(test_data['paths'])))
print("Overlap:", len(set(train_data['paths']) & set(test_data['paths']))) 

print("\nAugmented TESS Training DataFrame:")
train_data.head()

Unique train paths: 5040
Unique test paths: 280
Overlap: 0

Augmented TESS Training DataFrame:


,paths,emotions
0,../Data/TESS Toronto emotional speech set data...,pleasant_surprise
1,../Data/augmented_tess\OAF_gun_ps_augmented.wav,pleasant_surprise
2,../Data/TESS Toronto emotional speech set data...,pleasant_surprise
3,../Data/augmented_tess\OAF_shall_ps_augmented.wav,pleasant_surprise
4,../Data/TESS Toronto emotional speech set data...,neutral


## SAVEE Data

In [10]:
random.seed(42)      
np.random.seed(42)

augment = A.Compose([
    A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    A.PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    A.TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5)
]) 


augmented_dir = "../Data/augmented_savee"
os.makedirs(augmented_dir, exist_ok=True)


X_train_paths_savee, X_test_paths_savee, y_train_savee, y_test_savee = train_test_split(
    savee_file_paths, savee_labels, test_size=0.1, random_state=42, stratify=savee_labels
)

augmented_file_paths = []
augmented_labels = []

for file_path, label in zip(X_train_paths_savee, y_train_savee):
    try:
        audio, sr = librosa.load(file_path, sr=None)
        
        augmented_file_paths.append(file_path)
        augmented_labels.append(label)

        augmented_audio = augment(audio, sample_rate=sr)

        augmented_file_path = os.path.join(augmented_dir, os.path.basename(file_path).replace('.wav', '_augmented.wav'))
        sf.write(augmented_file_path, augmented_audio, sr)
        augmented_file_paths.append(augmented_file_path)
        augmented_labels.append(label)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

augmented_savee_train_data = pd.DataFrame({
    'paths': augmented_file_paths,
    'emotions': augmented_labels
})

savee_test_data = pd.DataFrame({
    'paths': X_test_paths_savee,
    'emotions': y_test_savee
})

print("Unique SAVEE train paths:", len(set(augmented_savee_train_data['paths'])))
print("Unique SAVEE test paths:", len(set(savee_test_data['paths'])))
print("Overlap:", len(set(augmented_savee_train_data['paths']) & set(savee_test_data['paths'])))  

print("\nAugmented SAVEE Training DataFrame:")
augmented_savee_train_data.head()

Unique SAVEE train paths: 552
Unique SAVEE test paths: 48
Overlap: 0

Augmented SAVEE Training DataFrame:


,paths,emotions
0,../Data/SAVEE data/SAVEE data\KL\su11.wav,pleasant_surprise
1,../Data/augmented_savee\su11_augmented.wav,pleasant_surprise
2,../Data/SAVEE data/SAVEE data\JK\n02.wav,neutral
3,../Data/augmented_savee\n02_augmented.wav,neutral
4,../Data/SAVEE data/SAVEE data\JK\su13.wav,pleasant_surprise


# Merge Datasets

In [11]:
train_emotion_data = pd.concat([train_data, augmented_savee_train_data], ignore_index=True)
test_emotion_data = pd.concat([test_data, savee_test_data], ignore_index=True)

print("Combined Training DataFrame:")
display(train_emotion_data.head())

print("\nCombined Test DataFrame:")
display(test_emotion_data.head())

print("\nTotal training samples:", len(train_emotion_data))
print("Total test samples:", len(test_emotion_data))
print("Overlap between train and test:", len(set(train_emotion_data['paths']) & set(test_emotion_data['paths'])))  

Combined Training DataFrame:


,paths,emotions
0,../Data/TESS Toronto emotional speech set data...,pleasant_surprise
1,../Data/augmented_tess\OAF_gun_ps_augmented.wav,pleasant_surprise
2,../Data/TESS Toronto emotional speech set data...,pleasant_surprise
3,../Data/augmented_tess\OAF_shall_ps_augmented.wav,pleasant_surprise
4,../Data/TESS Toronto emotional speech set data...,neutral



Combined Test DataFrame:


,paths,emotions
0,../Data/TESS Toronto emotional speech set data...,fear
1,../Data/TESS Toronto emotional speech set data...,neutral
2,../Data/TESS Toronto emotional speech set data...,pleasant_surprise
3,../Data/TESS Toronto emotional speech set data...,fear
4,../Data/TESS Toronto emotional speech set data...,happy



Total training samples: 5904
Total test samples: 328
Overlap between train and test: 0


# Handling incorrect class names

In [12]:
label_mapping = {
    'fear': 'fearful',
    'pleasant_surprised': 'pleasant_surprise'
}

train_emotion_data['emotions'] = train_emotion_data['emotions'].replace(label_mapping)
test_emotion_data['emotions'] = test_emotion_data['emotions'].replace(label_mapping)

print("Unique emotions in training data after spelling correction:", train_emotion_data['emotions'].unique())
print("Unique emotions in test data after spelling correction:", test_emotion_data['emotions'].unique())

Unique emotions in training data after spelling correction: ['pleasant_surprise' 'neutral' 'happy' 'angry' 'sad' 'fearful' 'disgust']
Unique emotions in test data after spelling correction: ['fearful' 'neutral' 'pleasant_surprise' 'happy' 'disgust' 'angry' 'sad']


# Imbalance Data Handling

In [13]:
random.seed(42)
np.random.seed(42)

balance_dir = "../Data/balanced_augmented"   
os.makedirs(balance_dir, exist_ok=True)

counts = train_emotion_data['emotions'].value_counts()
majority = counts.max()
median_count = int(counts.median())

min_target = max(median_count, int(0.6 * majority))  
max_target = majority                                  

print("Before balancing:\n", counts)

files_by_emotion = {emo: grp['paths'].tolist() for emo, grp in train_emotion_data.groupby('emotions')}
new_rows = []

def safe_augment_write(src_path, out_dir, idx_suffix):
    try:
        audio, sr = librosa.load(src_path, sr=None)
    except Exception as e:
        print(f"Could not load {src_path}: {e}")
        return None

    try:
        augmented_audio = augment(audio, sample_rate=sr)
    except Exception:
        noise = 0.0015 * np.random.randn(len(audio))   
        audio_noise = audio + noise

        n_steps = random.uniform(-1.0, 1.0)            
        try:
            audio_ps = librosa.effects.pitch_shift(audio_noise, sr, n_steps=n_steps)
        except Exception:
            audio_ps = audio_noise

        rate = random.uniform(0.95, 1.05)              
        try:
            augmented_audio = librosa.effects.time_stretch(audio_ps, rate)
        except Exception:
            augmented_audio = audio_ps

    base = os.path.splitext(os.path.basename(src_path))[0]
    out_fname = f"{base}_bal_{idx_suffix}.wav"
    out_path = os.path.join(out_dir, out_fname)
    
    j = 0
    while os.path.exists(out_path):
        j += 1
        out_path = os.path.join(out_dir, f"{base}_bal_{idx_suffix}_{j}.wav")

    try:
        sf.write(out_path, augmented_audio, sr)
        return out_path
    except Exception as e:
        print(f"Failed to write {out_path}: {e}")
        return None


for emo, paths in files_by_emotion.items():
    cur = len(paths)
    target = majority if cur < majority else cur

    need = target - cur
    if need <= 0:
        continue

    print(f"Generating {need} augmented files for '{emo}' (current {cur} -> target {target})")
    for i in range(need):
        src = random.choice(paths)   # Python random
        new_path = safe_augment_write(src, balance_dir, i)
        if new_path:
            new_rows.append({'paths': new_path, 'emotions': emo})


if new_rows:
    aug_df = pd.DataFrame(new_rows)
    train_emotion_data = pd.concat([train_emotion_data, aug_df], ignore_index=True)

print("After file-level augmentation counts:")
print(train_emotion_data['emotions'].value_counts())

classes = np.unique(train_emotion_data['emotions'])
class_weights_array = compute_class_weight(class_weight='balanced',
                                           classes=classes,
                                           y=train_emotion_data['emotions'].values)

class_weight_dict = dict(zip(classes, class_weights_array))
print("Class weights (label -> weight):")
print(class_weight_dict)

Before balancing:
 emotions
neutral              936
pleasant_surprise    828
happy                828
angry                828
sad                  828
fearful              828
disgust              828
Name: count, dtype: int64
Generating 108 augmented files for 'angry' (current 828 -> target 936)
Generating 108 augmented files for 'disgust' (current 828 -> target 936)
Generating 108 augmented files for 'fearful' (current 828 -> target 936)
Generating 108 augmented files for 'happy' (current 828 -> target 936)
Generating 108 augmented files for 'pleasant_surprise' (current 828 -> target 936)
Generating 108 augmented files for 'sad' (current 828 -> target 936)
After file-level augmentation counts:
emotions
pleasant_surprise    936
neutral              936
happy                936
angry                936
sad                  936
fearful              936
disgust              936
Name: count, dtype: int64
Class weights (label -> weight):
{'angry': 1.0, 'disgust': 1.0, 'fearful': 1.0, 'ha

## Saving the Train and Test Data

In [14]:
train_emotion_data.to_csv("../Data/train_emotion_data.csv", index=False)
test_emotion_data.to_csv("../Data/test_emotion_data.csv", index=False)

print("Train and Test CSV files saved successfully!")

Train and Test CSV files saved successfully!
